# 12 - Modelado con LSTM (enfoque global multi-embalse)
Tercer modelo del estudio. 
Se adopta el mismo **enfoque global** que en XGBoost: un único modelo aprende de los 17 embalses.

## 1. Configuración

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Concatenate
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

sys.path.append("..")

# Reproducibilidad
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

N_EJECUCIONES = 5           # nº de entrenamientos independientes (modificable)
SEEDS = [SEED + k for k in range(N_EJECUCIONES)]   # semillas: 42, 43, 44, 45, 46

DIR_PROCESSED = Path("../data/processed")
DIR_FIGURAS = Path("../outputs/figures")
DIR_RESULTADOS = Path("../outputs/tables")
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)
DIR_RESULTADOS.mkdir(parents=True, exist_ok=True)

TEST_INI = pd.Timestamp("2022-01-01")
HORIZONTES = [7, 30, 90]
LOOKBACK = 90          # días de historia que ve la red
PARAMETROS = ["amonio_mgl", "conductividad_uscm", "oxigeno_mgl",
              "ph", "temp_agua_c", "turbidez_ntu"]

df = pd.read_parquet(DIR_PROCESSED / "dataset_modelado.parquet")
df = df.sort_values(["ID_SAIH", "fecha"]).reset_index(drop=True)
experimentales = sorted(df["ID_SAIH"].unique())
print(f"Observaciones: {len(df):,} | embalses: {len(experimentales)} | lookback: {LOOKBACK} días")

Observaciones: 111,758 | embalses: 17 | lookback: 90 días


## 2. Variables
La LSTM recibe dos tipos de entrada:
- **Dinámicas**: varían cada día y forman la secuencia temporal.
- **Estáticas**: constantes por embalse.

In [2]:
CALIDAD = [f"cal_{p}" for p in PARAMETROS]

# Dinámicas crudas (sin lags ni medias móviles): la secuencia ya aporta esa información
DINAMICAS_BASE = [
    "pct_llenado", "aportacion_m3s", "salida_m3s",
    "aemet_temp_media_c", "aemet_temp_min_c", "aemet_temp_max_c",
    "aemet_precipitacion_mm", "aemet_humedad_pct", "et0_tipificada",
    "dia_anio_sin", "dia_anio_cos",
]

# Estáticas: capacidad + identificador one-hot del embalse
df_id = pd.get_dummies(df["ID_SAIH"], prefix="emb").astype(float)
COLS_ID = list(df_id.columns)
df = pd.concat([df, df_id], axis=1)
ESTATICAS = ["Capacidad_hm3"] + COLS_ID

ESCENARIOS = {
    "base": DINAMICAS_BASE,
    "base_calidad": DINAMICAS_BASE + CALIDAD,
}
for nombre, din in ESCENARIOS.items():
    print(f"{nombre}: {len(din)} dinámicas + {len(ESTATICAS)} estáticas")

base: 11 dinámicas + 18 estáticas
base_calidad: 17 dinámicas + 18 estáticas


## 3. Construcción de secuencias
Para cada embalse se generan ventanas deslizantes de `LOOKBACK` días consecutivos. Cada ventana (de un único embalse) constituye una muestra, cuyo objetivo es el porcentaje de llenado en el horizonte correspondiente.

In [3]:
def construir_secuencias(df, dinamicas, estaticas, horizonte, lookback, ids,
                         columnas_validez=None):
    """Devuelve Xd [n, lookback, n_din], Xs [n, n_estat], y [n] y meta.

    Una ventana es válida si no hay valores ausentes en `columnas_validez` (ni en el
    objetivo). Pasando el superconjunto de variables (las de base+calidad) como
    `columnas_validez` en ambos escenarios, los dos comparten EXACTAMENTE las mismas
    ventanas, de modo que la única diferencia entre ellos sean las variables de calidad."""
    objetivo = f"objetivo_h{horizonte}"
    val_cols = list(dict.fromkeys((columnas_validez or dinamicas) + [objetivo]))
    Xd, Xs, Y, META = [], [], [], []
    for emb in ids:
        g = df[df["ID_SAIH"] == emb].sort_values("fecha").reset_index(drop=True)
        vals = g[dinamicas].values
        chk = g[val_cols].values
        stat = g[estaticas].iloc[0].values
        obj = g[objetivo].values
        fechas = g["fecha"].values
        for i in range(len(g) - lookback + 1):
            t = i + lookback - 1
            if np.isnan(chk[i:i + lookback]).any() or np.isnan(obj[t]):
                continue
            Xd.append(vals[i:i + lookback]); Xs.append(stat); Y.append(obj[t])
            META.append((emb, fechas[t]))
    Xd = np.asarray(Xd, dtype="float32")
    Xs = np.asarray(Xs, dtype="float32")
    Y = np.asarray(Y, dtype="float32")
    meta = pd.DataFrame(META, columns=["ID_SAIH", "fecha_fin"])
    fin = pd.to_datetime(meta["fecha_fin"])
    # Embargo: una ventana es de train solo si su objetivo (fin + horizonte) NO cae en el test
    meta["es_train"] = fin < (TEST_INI - pd.Timedelta(days=horizonte))
    meta["es_test"] = fin >= TEST_INI
    # las ventanas cuyo fin está entre TEST_INI - h y TEST_INI (embargo) no son ni train ni test
    return Xd, Xs, Y, meta

# Prueba rápida con un horizonte para verificar dimensiones
Xd, Xs, Y, meta = construir_secuencias(df, ESCENARIOS["base_calidad"], ESTATICAS,
                                       30, LOOKBACK, experimentales,
                                       columnas_validez=ESCENARIOS["base_calidad"])
print(f"Ejemplo (h30, base+calidad): Xd {Xd.shape} | Xs {Xs.shape} | Y {Y.shape}")
print(f"Train/test: {meta['es_train'].value_counts().to_dict()}")

Ejemplo (h30, base+calidad): Xd (109643, 90, 17) | Xs (109643, 18) | Y (109643,)
Train/test: {True: 97233, False: 12410}


## 4. Métricas, normalización y arquitectura
Las métricas son las mismas que en los modelos anteriores.

In [4]:
def metricas(y_real, y_pred):
    y_real, y_pred = np.asarray(y_real), np.asarray(y_pred)
    m = ~(np.isnan(y_real) | np.isnan(y_pred))
    y_real, y_pred = y_real[m], y_pred[m]
    if len(y_real) == 0:
        return {"n": 0, "rmse": np.nan, "mae": np.nan, "nse": np.nan}
    err = y_real - y_pred
    sst = ((y_real - y_real.mean()) ** 2).sum()
    return {"n": len(y_real), "rmse": np.sqrt((err**2).mean()),
            "mae": np.abs(err).mean(),
            "nse": 1 - (err**2).sum() / sst if sst > 0 else np.nan}

def normalizar(Xd_tr, Xd_all, Xs_tr, Xs_all):
    """Ajusta media/std con TRAIN (dinámicas y estáticas por separado) y aplica a todo."""
    mu_d = Xd_tr.reshape(-1, Xd_tr.shape[-1]).mean(axis=0)
    sd_d = Xd_tr.reshape(-1, Xd_tr.shape[-1]).std(axis=0); sd_d[sd_d == 0] = 1.0
    mu_s = Xs_tr.mean(axis=0)
    sd_s = Xs_tr.std(axis=0); sd_s[sd_s == 0] = 1.0
    return ((Xd_all - mu_d) / sd_d).astype("float32"), ((Xs_all - mu_s) / sd_s).astype("float32")

def construir_modelo(lookback, n_din, n_estat, unidades=64, dropout=0.2):
    ent_seq = Input(shape=(lookback, n_din), name="secuencia")
    ent_est = Input(shape=(n_estat,), name="estaticas")
    
    # Prueba una capa
    x = LSTM(unidades)(ent_seq)

    # Prueba dos capas
    # x = LSTM(unidades, return_sequences=True)(ent_seq)
    # x = Dropout(dropout)(x)
    # x = LSTM(unidades)(x)
    
    x = Dropout(dropout)(x)
    x = Concatenate()([x, ent_est])
    x = Dense(32, activation="relu")(x)
    salida = Dense(1)(x)
    modelo = Model([ent_seq, ent_est], salida)
    modelo.compile(optimizer=Adam(learning_rate=0.0003), loss="mse")
    return modelo

## 5. Entrenamiento y evaluación
Para cada horizonte y escenario se construyen las secuencias, se normalizan con estadísticos de entrenamiento, se entrena la red con *early stopping* (reservando un 15 % del entrenamiento como validación) y se evalúa por embalse en el test.

In [5]:
EPOCAS = 60
BATCH = 256
PACIENCIA = 8
VALIDEZ = ESCENARIOS["base_calidad"]   # superconjunto que fija la muestra común

# ---- Persistencia: determinista, se calcula una sola vez (no depende de la ejecución) ----
resultados_persistencia = []
for h in HORIZONTES:
    Xd_c, Xs_c, Y_c, meta_c = construir_secuencias(
        df, ESCENARIOS["base_calidad"], ESTATICAS, h, LOOKBACK, experimentales,
        columnas_validez=VALIDEZ)
    te = meta_c["es_test"].values
    idx_llenado = ESCENARIOS["base_calidad"].index("pct_llenado")
    pers_pred = Xd_c[te, -1, idx_llenado]     # llenado del último día de cada ventana de test
    res_p = meta_c[te].copy(); res_p["real"] = Y_c[te]; res_p["pred"] = pers_pred
    for emb, gg in res_p.groupby("ID_SAIH"):
        resultados_persistencia.append({"embalse": emb, "horizonte": h,
                                         "escenario": "persistencia", **metricas(gg["real"], gg["pred"])})

# ---- Bucle de N ejecuciones independientes, con guardado incremental ----
for k, seed in enumerate(SEEDS, start=1):
    np.random.seed(seed)
    tf.random.set_seed(seed)

    resultados_run = list(resultados_persistencia)   # la persistencia es común a todas
    predicciones_run = []
    historiales_run = {}

    for h in HORIZONTES:
        for nombre, dinamicas in ESCENARIOS.items():
            Xd_e, Xs_e, Y_e, meta_e = construir_secuencias(
                df, dinamicas, ESTATICAS, h, LOOKBACK, experimentales, columnas_validez=VALIDEZ)
            tr = meta_e["es_train"].values
            te = meta_e["es_test"].values

            Xd_n, Xs_n = normalizar(Xd_e[tr], Xd_e, Xs_e[tr], Xs_e)

            modelo = construir_modelo(LOOKBACK, Xd_e.shape[-1], Xs_e.shape[-1])
            es = EarlyStopping(monitor="val_loss", patience=PACIENCIA, restore_best_weights=True)
            hist = modelo.fit([Xd_n[tr], Xs_n[tr]], Y_e[tr],
                              validation_split=0.15, epochs=EPOCAS, batch_size=BATCH,
                              callbacks=[es], verbose=0)
            historiales_run[(h, nombre)] = hist.history

            pred = modelo.predict([Xd_n[te], Xs_n[te]], verbose=0).ravel()
            res_te = meta_e[te].copy(); res_te["real"] = Y_e[te]; res_te["pred"] = pred
            for emb, gg in res_te.groupby("ID_SAIH"):
                resultados_run.append({"embalse": emb, "horizonte": h, "escenario": nombre,
                                       **metricas(gg["real"], gg["pred"])})
            predicciones_run.append(res_te.assign(horizonte=h, escenario=nombre))
            print(f"  [ejec {k}] h{h} {nombre}: {len(hist.history['loss'])} épocas | "
                  f"val_loss={hist.history['val_loss'][-1]:.3f} | muestras={len(Y_e)}")

    # Guardado incremental: cada ejecución guarda lo suyo nada más terminar
    pd.DataFrame(resultados_run).assign(ejecucion=k).to_parquet(
        DIR_RESULTADOS / f"lstm_metricas_run{k}.parquet", index=False)
    pd.concat(predicciones_run).assign(ejecucion=k).to_parquet(
        DIR_RESULTADOS / f"lstm_predicciones_run{k}.parquet", index=False)

    # Curvas de entrenamiento de esta ejecución
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, h in zip(axes, HORIZONTES):
        hh = historiales_run[(h, "base_calidad")]
        ax.plot(hh["loss"], label="entrenamiento")
        ax.plot(hh["val_loss"], label="validación")
        ax.set_title(f"Horizonte {h} días"); ax.set_xlabel("época"); ax.set_ylabel("MSE"); ax.legend()
    plt.tight_layout()
    plt.savefig(DIR_FIGURAS / f"5_5_lstm_curvas_run{k}.png", dpi=150, bbox_inches="tight")
    plt.close()

    print(f"  Ejecución {k}/{N_EJECUCIONES} (seed={seed}) guardada\n")

print("Todas las ejecuciones completadas.")

  [ejec 1] h7 base: 13 épocas | val_loss=78.404 | muestras=110034
  [ejec 1] h7 base_calidad: 14 épocas | val_loss=109.119 | muestras=110034
  [ejec 1] h30 base: 13 épocas | val_loss=156.863 | muestras=109643
  [ejec 1] h30 base_calidad: 13 épocas | val_loss=275.649 | muestras=109643
  [ejec 1] h90 base: 14 épocas | val_loss=308.142 | muestras=108623
  [ejec 1] h90 base_calidad: 13 épocas | val_loss=300.298 | muestras=108623
  Ejecución 1/5 (seed=42) guardada

  [ejec 2] h7 base: 12 épocas | val_loss=105.847 | muestras=110034
  [ejec 2] h7 base_calidad: 13 épocas | val_loss=167.299 | muestras=110034
  [ejec 2] h30 base: 13 épocas | val_loss=145.722 | muestras=109643
  [ejec 2] h30 base_calidad: 12 épocas | val_loss=217.096 | muestras=109643
  [ejec 2] h90 base: 13 épocas | val_loss=344.210 | muestras=108623
  [ejec 2] h90 base_calidad: 18 épocas | val_loss=321.092 | muestras=108623
  Ejecución 2/5 (seed=43) guardada

  [ejec 3] h7 base: 13 épocas | val_loss=100.659 | muestras=110034
  

In [6]:
# --- Agregación de las N ejecuciones: mediana y rango ---
dfs = [pd.read_parquet(DIR_RESULTADOS / f"lstm_metricas_run{k}.parquet")
       for k in range(1, N_EJECUCIONES + 1)]
todas = pd.concat(dfs, ignore_index=True)

# Mediana entre embalses por ejecución, luego mediana y rango entre ejecuciones
med_por_ejec = (todas.groupby(["horizonte", "escenario", "ejecucion"])[["rmse", "mae", "nse"]]
                .median().reset_index())

def fmt(m, mn, mx):
    return f"{m:.3f} ({mn:.3f} a {mx:.3f})"

filas = []
for (h, esc), g in med_por_ejec.groupby(["horizonte", "escenario"]):
    fila = {"Horizonte": h, "Escenario": esc}
    for met, et in [("rmse", "RMSE"), ("mae", "MAE"), ("nse", "NSE")]:
        # la persistencia es determinista: mostrar solo la mediana sin rango
        if esc == "persistencia":
            fila[et] = f"{g[met].median():.3f}"
        else:
            fila[et] = fmt(g[met].median(), g[met].min(), g[met].max())
    filas.append(fila)

agregado = pd.DataFrame(filas)
agregado.to_parquet(DIR_RESULTADOS / "lstm_metricas_agregado.parquet", index=False)
print("=== Medianas y rango de las N ejecuciones ===")
print(agregado.to_string(index=False))

=== Medianas y rango de las N ejecuciones ===
 Horizonte    Escenario                     RMSE                   MAE                      NSE
         7         base    4.949 (4.594 a 5.109) 3.809 (3.579 a 3.995)    0.565 (0.524 a 0.581)
         7 base_calidad    4.995 (4.905 a 5.275) 3.844 (3.733 a 4.026)    0.562 (0.503 a 0.567)
         7 persistencia                    4.420                 2.877                    0.634
        30         base    7.453 (6.718 a 7.661) 5.863 (5.474 a 6.222) -0.153 (-0.168 a -0.036)
        30 base_calidad    8.116 (7.455 a 8.240) 6.588 (5.936 a 6.966) -0.159 (-0.196 a -0.062)
        30 persistencia                    8.122                 6.062                   -0.410
        90         base   9.648 (9.454 a 11.907) 7.653 (7.184 a 9.268) -0.190 (-0.336 a -0.077)
        90 base_calidad 11.278 (10.293 a 11.693) 7.852 (7.576 a 9.237) -0.344 (-0.424 a -0.234)
        90 persistencia                   11.129                 8.974                   -

## 6. Comparación con la persistencia

## 7. Curvas de entrenamiento
Se revisan las curvas de pérdida (entrenamiento y validación) para comprobar que el entrenamiento converge y que el *early stopping* actúa antes del sobreajuste.

# Tablas de resultados para la memoria

In [7]:
# --- Tablas del LSTM: agregadas por embalse (texto) + individuales por run (anexo) ---
NOMBRES = {
    "E002": "E002 · Os Peares", "E008": "E008 · Fuente del Azufre", "E009": "E009 · Montearenas",
    "E011": "E011 · Peñarrubia", "E025": "E025 · Leboreiro Mao", "E026": "E026 · Edrada Mao",
    "E027": "E027 · San Esteban", "E028": "E028 · Vilasouto", "E029": "E029 · San Pedro",
    "E030": "E030 · Velle", "E031": "E031 · Castrelo", "E033": "E033 · Frieira",
    "E07A": "E07A · Bárcena", "E32A": "E32A · Albarellos", "E35A": "E35A · Conchas",
    "E570": "E570 · Santiago", "E571": "E571 · Pumares",
}
ESC = {"base": "Base", "base_calidad": "Base + calidad", "persistencia": "Persistencia"}

# Cargar las N ejecuciones guardadas
dfs = [pd.read_parquet(DIR_RESULTADOS / f"lstm_metricas_run{k}.parquet")
       for k in range(1, N_EJECUCIONES + 1)]
todas = pd.concat(dfs, ignore_index=True)

def fmt(m, mn, mx):
    return f"{m:.3f} ({mn:.3f} a {mx:.3f})"

# ---- 3 TABLAS AGREGADAS por embalse (mediana y rango de las N ejecuciones) -> TEXTO ----
print("="*70)
print("TABLAS AGREGADAS POR EMBALSE (para el texto)")
print("="*70)
for h in HORIZONTES:
    sub = todas[todas["horizonte"] == h]
    filas = []
    for emb in sorted(sub["embalse"].unique()):
        fila = {"Embalse": NOMBRES.get(emb, emb)}
        for esc in ["base", "base_calidad", "persistencia"]:
            vals = sub[(sub["embalse"] == emb) & (sub["escenario"] == esc)]["nse"]
            if esc == "persistencia":
                fila[ESC[esc]] = f"{vals.median():.3f}"   # determinista, sin rango
            else:
                fila[ESC[esc]] = fmt(vals.median(), vals.min(), vals.max())
        filas.append(fila)
    # Fila de mediana global (mediana de las medianas por embalse)
    tabla = pd.DataFrame(filas)
    tabla.to_csv(DIR_RESULTADOS / f"lstm_nse_embalse_h{h}_agregado.csv", index=False)
    print(f"\n--- NSE por embalse, horizonte {h} días (mediana y rango de {N_EJECUCIONES} ejec.) ---")
    print(tabla.to_markdown(index=False))

# ---- 15 TABLAS INDIVIDUALES por run y horizonte -> ANEXO ----
for k in range(1, N_EJECUCIONES + 1):
    run = todas[todas["ejecucion"] == k]
    for h in HORIZONTES:
        t = (run[run["horizonte"] == h]
             .pivot_table(index="embalse", columns="escenario", values="nse")
             .round(3))
        t = t[["base", "base_calidad", "persistencia"]]
        t.loc["Mediana"] = t.median().round(3)
        t.index = [NOMBRES.get(e, e) for e in t.index]
        t.columns = [ESC[c] for c in t.columns]
        t.to_csv(DIR_RESULTADOS / f"lstm_nse_run{k}_h{h}.csv")

print(f"\n\nGuardados: 3 CSV agregados (texto) + {N_EJECUCIONES*len(HORIZONTES)} CSV individuales (anexo)")

TABLAS AGREGADAS POR EMBALSE (para el texto)

--- NSE por embalse, horizonte 7 días (mediana y rango de 5 ejec.) ---
| Embalse                  | Base                       | Base + calidad              |   Persistencia |
|:-------------------------|:---------------------------|:----------------------------|---------------:|
| E002 · Os Peares         | 0.426 (0.423 a 0.498)      | 0.456 (0.416 a 0.499)       |          0.634 |
| E008 · Fuente del Azufre | -2.295 (-2.582 a -1.874)   | -3.097 (-3.627 a -1.812)    |         -0.608 |
| E009 · Montearenas       | 0.565 (0.524 a 0.581)      | 0.562 (0.503 a 0.567)       |          0.481 |
| E011 · Peñarrubia        | 0.684 (0.651 a 0.719)      | 0.673 (0.663 a 0.704)       |          0.778 |
| E025 · Leboreiro Mao     | 0.772 (0.767 a 0.788)      | 0.780 (0.776 a 0.795)       |          0.791 |
| E026 · Edrada Mao        | 0.847 (0.843 a 0.868)      | 0.858 (0.838 a 0.861)       |          0.881 |
| E027 · San Esteban       | 0.403 (0.358 a

In [8]:
# Conteo: ¿en cuántos embalses la MEDIANA del LSTM (base) supera a la persistencia?
dfs = [pd.read_parquet(DIR_RESULTADOS / f"lstm_metricas_run{k}.parquet")
       for k in range(1, N_EJECUCIONES + 1)]
todas = pd.concat(dfs, ignore_index=True)

# Mediana por embalse de las N ejecuciones (base y persistencia)
med = (todas.groupby(["embalse", "horizonte", "escenario"])["nse"]
       .median().reset_index())

for h in HORIZONTES:
    piv = med[med["horizonte"] == h].pivot_table(index="embalse", columns="escenario", values="nse")
    supera = (piv["base"] > piv["persistencia"]).sum()
    print(f"h={h}: LSTM base (mediana) supera a persistencia en {supera}/17 embalses")

h=7: LSTM base (mediana) supera a persistencia en 6/17 embalses
h=30: LSTM base (mediana) supera a persistencia en 11/17 embalses
h=90: LSTM base (mediana) supera a persistencia en 13/17 embalses


In [9]:
# Diagnóstico de casos extremos en el LSTM (usa run1 como ejecución representativa)
pred = pd.read_parquet(DIR_RESULTADOS / "lstm_predicciones_run1.parquet")

for emb in ["E008", "E571"]:
    print(f"\n{'='*55}\n{emb}\n{'='*55}")
    for h in HORIZONTES:
        sub = pred[(pred["horizonte"] == h) & (pred["escenario"] == "base") &
                   (pred["ID_SAIH"] == emb)]
        if len(sub) == 0:
            continue
        real, pr = sub["real"], sub["pred"]
        mae = (real - pr).abs().mean()
        rmse = ((real - pr)**2).mean()**0.5
        std_real = real.std()
        print(f"  h{h}: std_real={std_real:.2f}, MAE={mae:.2f}, RMSE={rmse:.2f}, "
              f"rango_real=[{real.min():.1f}, {real.max():.1f}]")


E008
  h7: std_real=1.58, MAE=2.25, RMSE=3.00, rango_real=[23.2, 34.0]
  h30: std_real=1.61, MAE=4.07, RMSE=5.10, rango_real=[23.2, 34.0]
  h90: std_real=1.60, MAE=6.90, RMSE=8.41, rango_real=[23.2, 34.0]

E571
  h7: std_real=3.53, MAE=10.49, RMSE=10.97, rango_real=[54.6, 96.6]
  h30: std_real=3.58, MAE=14.53, RMSE=14.78, rango_real=[54.6, 96.6]
  h90: std_real=3.71, MAE=19.19, RMSE=19.40, rango_real=[54.6, 96.6]
